# Insurance Claims Automation Capstone Lab Notebook
## LangChain + RAG + LangGraph Multi-Agent + MCP + FastAPI + LangSmith + Azure Deployment

This beginner-friendly notebook converts the Capstone Azure Deployment Guide into a hands-on lab.

Participants will build and deploy an **End-to-End Insurance Claims Automation Platform** using Python, FastAPI, RAG, LangGraph, MCP-style tools, LangSmith, Docker and Azure App Service.

## Capstone Objective

The platform will:

- Accept insurance claim documents or claim data.
- Extract and validate structured claim information.
- Use RAG to answer policy and coverage questions.
- Use Generative AI to summarize claim and policy context.
- Use LangGraph multi-agent workflow for policy verification, fraud review, routing and human review.
- Expose MCP-style tools for policy lookup, claim validation and document analysis.
- Provide FastAPI endpoints for UiPath integration.
- Use LangSmith for tracing and debugging.
- Deploy securely on Azure Cloud.

## Target Architecture

```text
Claim Portal / UiPath / API Client
        ↓
FastAPI Claim Processing API
        ↓
Python Validation Layer
        ↓
RAG Policy Knowledge Assistant
        ↓
LangGraph Multi-Agent Workflow
        ↓
MCP-Style Tool Layer
        ├── Policy Lookup Tool
        ├── Claim Validation Tool
        ├── Document Analysis Tool
        └── Mock UiPath Trigger Tool
        ↓
Routing Decision
        ├── Approved
        ├── Rework Required
        ├── Human Review
        └── Rejected
        ↓
Customer Email + Audit Report
        ↓
LangSmith Tracing
        ↓
Azure App Service
```

## How to Use This Notebook

1. Run setup cells to generate the project files.
2. Open generated folder in VS Code.
3. Create `.env` from `.env.example`.
4. Install requirements.
5. Run FastAPI locally.
6. Test Swagger UI.
7. Build Docker image.
8. Push Docker image to Azure Container Registry.
9. Deploy to Azure App Service.
10. Test the deployed project link.

Shell commands should be run in VS Code Terminal, PowerShell, macOS Terminal, or Azure Cloud Shell.

In [1]:
from pathlib import Path
import json

PROJECT_DIR = Path("insurance-claims-capstone")
PROJECT_DIR.mkdir(exist_ok=True)

print("Project folder created:", PROJECT_DIR.resolve())

Project folder created: /Users/surendra/Desktop/Allianz_Agentic_AI/insurance-claims-capstone


# Step 1: VS Code Environment Setup

Open VS Code and select:

```text
File → Open Folder → insurance-claims-capstone
```

Create virtual environment.

Mac/Linux:

```bash
python3 -m venv .venv
source .venv/bin/activate
```

Windows PowerShell:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

When terminal shows `(.venv)`, your project environment is active.

# Step 2: API Key and LangSmith Setup

Create `.env` locally from `.env.example`.

Required values:

```text
OPENROUTER_API_KEY=your_openrouter_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini

LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_langsmith_api_key_here
LANGSMITH_PROJECT=insurance-claims-capstone
```

Do not commit `.env` to GitHub. In Azure, use App Settings or Azure Key Vault.

In [1]:
from pathlib import Path

PROJECT_DIR = Path("insurance-claims-capstone")

env_example = '''OPENROUTER_API_KEY=your_openrouter_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini

LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your_langsmith_api_key_here
LANGSMITH_PROJECT=insurance-claims-capstone

APP_ENV=local
'''

gitignore = '''.env
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
'''

(PROJECT_DIR / ".env.example").write_text(env_example, encoding="utf-8")
(PROJECT_DIR / ".gitignore").write_text(gitignore, encoding="utf-8")

print("Created .env.example and .gitignore")

Created .env.example and .gitignore


# Step 3: Create requirements.txt

This file contains Python libraries required for the capstone.

In [2]:
requirements = '''fastapi
uvicorn
openai
python-dotenv
pydantic
langgraph
langchain
langchain-openai
langsmith
requests
pandas
numpy
scikit-learn
'''

(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print("Created requirements.txt")

Created requirements.txt


Install dependencies:

```bash
cd insurance-claims-capstone
pip install -r requirements.txt
```

Quick test:

```bash
python -c "import fastapi, openai, langgraph; print('Setup working')"
```

# Step 4: Create Data Models

`models.py` validates input data for FastAPI endpoints.

In [3]:
models_py = '''from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field


class ClaimRequest(BaseModel):
    claim_id: str = Field(..., description="Unique claim identifier")
    policy_number: str = Field(..., description="Insurance policy number")
    customer_name: str = Field(..., description="Customer full name")
    claim_type: str = Field(..., description="Health, Motor, Travel or Property")
    claim_amount: float = Field(..., description="Requested claim amount")
    claim_description: Optional[str] = ""
    submitted_documents: List[str] = []


class PolicyQuestionRequest(BaseModel):
    question: str
    policy_number: Optional[str] = None


class AgentResponse(BaseModel):
    status: str
    result: Dict[str, Any]
'''

(PROJECT_DIR / "models.py").write_text(models_py, encoding="utf-8")
print("Created models.py")

Created models.py


# Step 5: Create MCP-Style Tools

MCP-style tools are controlled business functions that an AI agent can call.

Tools in this lab:

| Tool | Purpose |
|---|---|
| `policy_lookup_tool` | Finds policy details |
| `claim_validation_tool` | Validates claim against policy |
| `document_checklist_tool` | Returns required documents |
| `document_analysis_tool` | Finds missing documents |
| `mock_uipath_trigger_tool` | Simulates UiPath downstream automation |

In [4]:
mcp_tools_py = 'from typing import Dict, Any, List\n\npolicy_database = {\n    "POL1001": {\n        "customer_name": "Rajesh Sharma",\n        "policy_type": "Health",\n        "policy_status": "Active",\n        "premium_status": "Paid",\n        "coverage_amount": 500000\n    },\n    "POL2002": {\n        "customer_name": "Meena Joshi",\n        "policy_type": "Motor",\n        "policy_status": "Active",\n        "premium_status": "Paid",\n        "coverage_amount": 300000\n    },\n    "POL3003": {\n        "customer_name": "Amit Verma",\n        "policy_type": "Travel",\n        "policy_status": "Active",\n        "premium_status": "Pending",\n        "coverage_amount": 100000\n    }\n}\n\nrequired_documents = {\n    "Health": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],\n    "Motor": ["Repair Estimate", "Vehicle Photos", "Police Report", "Driving License", "RC Copy"],\n    "Travel": ["Boarding Pass", "Airline Delay Certificate", "Baggage Delay Report", "Claim Form"],\n    "Property": ["Damage Photos", "Repair Estimate", "Ownership Proof", "Claim Form"]\n}\n\ndef policy_lookup_tool(policy_number: str) -> Dict[str, Any]:\n    return policy_database.get(policy_number, {"error": "Policy not found"})\n\ndef document_checklist_tool(claim_type: str) -> List[str]:\n    return required_documents.get(claim_type, [])\n\ndef document_analysis_tool(claim_type: str, submitted_documents: List[str]) -> Dict[str, Any]:\n    required = document_checklist_tool(claim_type)\n    missing = [doc for doc in required if doc not in submitted_documents]\n    return {\n        "required_documents": required,\n        "submitted_documents": submitted_documents,\n        "missing_documents": missing,\n        "document_status": "Complete" if not missing else "Missing Documents"\n    }\n\ndef claim_validation_tool(claim: Dict[str, Any]) -> Dict[str, Any]:\n    policy = policy_lookup_tool(claim.get("policy_number", ""))\n    if "error" in policy:\n        return {"valid": False, "reason": "Policy not found", "next_action": "Route to policy exception queue"}\n    if policy["policy_status"] != "Active":\n        return {"valid": False, "reason": "Policy inactive", "next_action": "Reject or request policy review"}\n    if policy["premium_status"] != "Paid":\n        return {"valid": False, "reason": "Premium pending", "next_action": "Request premium payment verification"}\n    if claim.get("claim_amount", 0) > policy["coverage_amount"]:\n        return {"valid": False, "reason": "Claim exceeds coverage", "next_action": "Route to human review"}\n    return {"valid": True, "reason": "Claim is within active policy coverage", "next_action": "Proceed to document and risk review"}\n\ndef mock_uipath_trigger_tool(process_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:\n    return {\n        "trigger_status": "success",\n        "process_name": process_name,\n        "job_id": "UIPATH-MOCK-JOB-1001",\n        "payload": payload,\n        "message": "UiPath robot trigger simulated successfully"\n    }\n\ndef discover_tools() -> Dict[str, Any]:\n    return {\n        "policy_lookup_tool": "Looks up policy details by policy number",\n        "claim_validation_tool": "Validates claim against policy coverage and premium status",\n        "document_checklist_tool": "Returns required documents by claim type",\n        "document_analysis_tool": "Checks submitted versus required documents",\n        "mock_uipath_trigger_tool": "Simulates downstream UiPath robot trigger"\n    }\n'
(PROJECT_DIR / "mcp_tools.py").write_text(mcp_tools_py, encoding="utf-8")
print("Created mcp_tools.py")

Created mcp_tools.py


Test MCP tools:

```bash
python -c "from mcp_tools import policy_lookup_tool; print(policy_lookup_tool('POL1001'))"
```

# Step 6: Create RAG Service

RAG means **Retrieval-Augmented Generation**.

In this beginner lab:

- Policy clauses are stored in a Python list.
- TF-IDF simulates vector retrieval.
- LLM generates an answer using retrieved context.

In [5]:
rag_service_py = 'import os\nfrom typing import Dict, Any, List\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nload_dotenv()\n\nOPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")\nOPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")\nOPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")\n\nclient = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)\n\npolicy_clauses = [\n    {"title": "Health Secure Plus - Hospitalization", "text": "Health Secure Plus covers hospitalization expenses up to INR 5,00,000. Room rent is covered up to INR 5,000 per day."},\n    {"title": "Health Secure Plus - Pre and Post Hospitalization", "text": "Pre-hospitalization expenses are covered for 30 days. Post-hospitalization expenses are covered for 60 days."},\n    {"title": "Health Secure Plus - Exclusions", "text": "Cosmetic treatment, non-prescribed medicines and experimental procedures are excluded from coverage."},\n    {"title": "Motor Policy - Own Damage", "text": "Comprehensive motor insurance covers own damage due to accident, theft, fire and natural calamities."},\n    {"title": "Motor Claim Documents", "text": "Motor claim documents include repair estimate, vehicle photos, police report, driving license and RC copy."},\n    {"title": "Travel Insurance - Baggage Delay", "text": "Baggage delay is covered if baggage is delayed by more than 12 hours. Required documents include boarding pass and airline delay certificate."}\n]\n\ntexts = [item["text"] for item in policy_clauses]\nvectorizer = TfidfVectorizer()\nvectors = vectorizer.fit_transform(texts)\n\ndef call_llm(prompt: str, temperature: float = 0.2) -> str:\n    response = client.chat.completions.create(\n        model=OPENROUTER_MODEL,\n        messages=[\n            {"role": "system", "content": "You are an insurance policy assistant. Answer factually using provided context."},\n            {"role": "user", "content": prompt}\n        ],\n        temperature=temperature\n    )\n    return response.choices[0].message.content\n\ndef retrieve_policy_context(question: str, top_k: int = 3) -> List[Dict[str, Any]]:\n    query_vector = vectorizer.transform([question])\n    similarities = cosine_similarity(query_vector, vectors).flatten()\n    ranked = similarities.argsort()[::-1][:top_k]\n    return [{"title": policy_clauses[i]["title"], "text": policy_clauses[i]["text"], "score": float(similarities[i])} for i in ranked]\n\ndef answer_policy_question(question: str) -> Dict[str, Any]:\n    retrieved = retrieve_policy_context(question)\n    context = "\\n\\n".join([f"Title: {item[\'title\']}\\nText: {item[\'text\']}" for item in retrieved])\n    prompt = (\n        "Answer the insurance policy question using only the context below.\\\\n"\n        "Rules:\\\\n"\n        "- Do not invent policy details.\\\\n"\n        "- If information is missing, say it is not available in the provided context.\\\\n"\n        "- Mention source titles briefly.\\\\n\\\\n"\n        f"Context:\\\\n{context}\\\\n\\\\nQuestion:\\\\n{question}"\n    )\n    answer = call_llm(prompt)\n    return {"question": question, "answer": answer, "sources": [item["title"] for item in retrieved]}\n'
(PROJECT_DIR / "rag_service.py").write_text(rag_service_py, encoding="utf-8")
print("Created rag_service.py")

Created rag_service.py


Test RAG after configuring `.env`:

```bash
python -c "from rag_service import answer_policy_question; print(answer_policy_question('Is room rent covered?'))"
```

# Step 7: Create LangGraph Multi-Agent Workflow

This workflow processes a claim using multiple agents:

```text
Claim Intake → Policy Verification → Document Validation → Fraud Review → Routing → Customer Message → Audit → UiPath Trigger
```

In [6]:
langgraph_agent_py = 'import os\nimport json\nimport time\nfrom typing import TypedDict, Dict, Any, List\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\nfrom langgraph.graph import StateGraph, START, END\n\nfrom mcp_tools import policy_lookup_tool, claim_validation_tool, document_analysis_tool, mock_uipath_trigger_tool\n\nload_dotenv()\n\nOPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")\nOPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")\nOPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")\n\nclient = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)\n\ndef call_llm(prompt: str, system_message: str = "You are an insurance claims automation assistant.") -> str:\n    response = client.chat.completions.create(\n        model=OPENROUTER_MODEL,\n        messages=[\n            {"role": "system", "content": system_message},\n            {"role": "user", "content": prompt}\n        ],\n        temperature=0.2\n    )\n    return response.choices[0].message.content\n\nclass ClaimState(TypedDict, total=False):\n    claim: Dict[str, Any]\n    claim_summary: str\n    policy_result: Dict[str, Any]\n    validation_result: Dict[str, Any]\n    document_result: Dict[str, Any]\n    risk_level: str\n    route: str\n    customer_message: str\n    audit_report: Dict[str, Any]\n    uipath_result: Dict[str, Any]\n    logs: List[str]\n\ndef claim_intake_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    prompt = "Summarize this insurance claim in 4-5 lines for a claims officer.\\\\n\\\\nClaim:\\\\n" + json.dumps(claim, indent=2)\n    summary = call_llm(prompt)\n    logs = state.get("logs", [])\n    logs.append("Claim intake agent completed.")\n    return {**state, "claim_summary": summary, "logs": logs}\n\ndef policy_verification_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    policy_result = policy_lookup_tool(claim["policy_number"])\n    validation_result = claim_validation_tool(claim)\n    logs = state.get("logs", [])\n    logs.append("Policy verification agent completed.")\n    return {**state, "policy_result": policy_result, "validation_result": validation_result, "logs": logs}\n\ndef document_validation_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    document_result = document_analysis_tool(claim["claim_type"], claim.get("submitted_documents", []))\n    logs = state.get("logs", [])\n    logs.append("Document validation agent completed.")\n    return {**state, "document_result": document_result, "logs": logs}\n\ndef fraud_risk_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    description = claim.get("claim_description", "").lower()\n    amount = claim.get("claim_amount", 0)\n    risk_level = "Low"\n    if amount > 300000:\n        risk_level = "High"\n    elif "unclear" in description or "midnight" in description or "duplicate" in description:\n        risk_level = "Medium"\n    logs = state.get("logs", [])\n    logs.append("Fraud risk review agent completed.")\n    return {**state, "risk_level": risk_level, "logs": logs}\n\ndef routing_agent(state: ClaimState) -> ClaimState:\n    validation = state["validation_result"]\n    documents = state["document_result"]\n    risk = state["risk_level"]\n    if not validation["valid"]:\n        route = "Rejected / Policy Exception"\n    elif documents["document_status"] == "Missing Documents":\n        route = "Rework Required"\n    elif risk == "High":\n        route = "Human Review"\n    else:\n        route = "Approved for Processing"\n    logs = state.get("logs", [])\n    logs.append("Routing agent completed.")\n    return {**state, "route": route, "logs": logs}\n\ndef customer_message_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    prompt = (\n        "Create a professional customer email for this insurance claim.\\\\n\\\\n"\n        f"Claim ID: {claim[\'claim_id\']}\\\\n"\n        f"Customer: {claim[\'customer_name\']}\\\\n"\n        f"Route: {state[\'route\']}\\\\n"\n        f"Document Status: {state[\'document_result\'][\'document_status\']}\\\\n"\n        f"Validation Result: {state[\'validation_result\'][\'reason\']}\\\\n\\\\n"\n        "Keep it polite, clear and concise."\n    )\n    message = call_llm(prompt)\n    logs = state.get("logs", [])\n    logs.append("Customer message agent completed.")\n    return {**state, "customer_message": message, "logs": logs}\n\ndef audit_agent(state: ClaimState) -> ClaimState:\n    claim = state["claim"]\n    audit = {\n        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "claim_id": claim["claim_id"],\n        "policy_number": claim["policy_number"],\n        "policy_result": state["policy_result"],\n        "validation_result": state["validation_result"],\n        "document_result": state["document_result"],\n        "risk_level": state["risk_level"],\n        "route": state["route"],\n        "logs": state.get("logs", [])\n    }\n    logs = state.get("logs", [])\n    logs.append("Audit agent completed.")\n    return {**state, "audit_report": audit, "logs": logs}\n\ndef uipath_trigger_agent(state: ClaimState) -> ClaimState:\n    route = state["route"]\n    if route == "Approved for Processing":\n        process_name = "Claim_Auto_Processing_Robot"\n    elif route == "Rework Required":\n        process_name = "Document_Rework_Robot"\n    elif route == "Human Review":\n        process_name = "Human_Review_Robot"\n    else:\n        process_name = "Policy_Exception_Robot"\n    uipath_result = mock_uipath_trigger_tool(process_name, {"claim_id": state["claim"]["claim_id"], "route": route})\n    logs = state.get("logs", [])\n    logs.append("UiPath trigger agent completed.")\n    return {**state, "uipath_result": uipath_result, "logs": logs}\n\nworkflow = StateGraph(ClaimState)\nworkflow.add_node("claim_intake", claim_intake_agent)\nworkflow.add_node("policy_verification", policy_verification_agent)\nworkflow.add_node("document_validation", document_validation_agent)\nworkflow.add_node("fraud_risk", fraud_risk_agent)\nworkflow.add_node("routing", routing_agent)\nworkflow.add_node("customer_message", customer_message_agent)\nworkflow.add_node("audit", audit_agent)\nworkflow.add_node("uipath_trigger", uipath_trigger_agent)\n\nworkflow.add_edge(START, "claim_intake")\nworkflow.add_edge("claim_intake", "policy_verification")\nworkflow.add_edge("policy_verification", "document_validation")\nworkflow.add_edge("document_validation", "fraud_risk")\nworkflow.add_edge("fraud_risk", "routing")\nworkflow.add_edge("routing", "customer_message")\nworkflow.add_edge("customer_message", "audit")\nworkflow.add_edge("audit", "uipath_trigger")\nworkflow.add_edge("uipath_trigger", END)\n\nclaim_agent_app = workflow.compile()\n\ndef process_claim_with_agent(claim: Dict[str, Any]) -> Dict[str, Any]:\n    final_state = claim_agent_app.invoke({"claim": claim, "logs": []})\n    return {\n        "claim_id": claim["claim_id"],\n        "claim_summary": final_state.get("claim_summary"),\n        "policy_result": final_state.get("policy_result"),\n        "validation_result": final_state.get("validation_result"),\n        "document_result": final_state.get("document_result"),\n        "risk_level": final_state.get("risk_level"),\n        "route": final_state.get("route"),\n        "customer_message": final_state.get("customer_message"),\n        "audit_report": final_state.get("audit_report"),\n        "uipath_result": final_state.get("uipath_result"),\n        "logs": final_state.get("logs")\n    }\n'
(PROJECT_DIR / "langgraph_agent.py").write_text(langgraph_agent_py, encoding="utf-8")
print("Created langgraph_agent.py")

Created langgraph_agent.py


# Step 8: Create FastAPI App

Endpoints:

| Endpoint | Purpose |
|---|---|
| `GET /` | Home |
| `GET /health` | Health check |
| `GET /tools` | MCP-style tool discovery |
| `POST /policy/ask` | RAG policy question |
| `POST /claim/process` | Full claim automation workflow |

In [7]:
app_py = 'from fastapi import FastAPI\nfrom models import ClaimRequest, PolicyQuestionRequest\nfrom langgraph_agent import process_claim_with_agent\nfrom rag_service import answer_policy_question\nfrom mcp_tools import discover_tools\n\napp = FastAPI(\n    title="Insurance Claims Automation Capstone API",\n    description="LangChain + RAG + LangGraph Multi-Agent + MCP + FastAPI + LangSmith + Azure Deployment",\n    version="1.0.0"\n)\n\n@app.get("/")\ndef home():\n    return {"message": "Insurance Claims Automation Capstone API is running", "docs": "/docs"}\n\n@app.get("/health")\ndef health_check():\n    return {"status": "healthy", "service": "insurance-claims-capstone"}\n\n@app.get("/tools")\ndef list_tools():\n    return {"available_tools": discover_tools()}\n\n@app.post("/policy/ask")\ndef ask_policy_question(request: PolicyQuestionRequest):\n    result = answer_policy_question(request.question)\n    return {"status": "success", "result": result}\n\n@app.post("/claim/process")\ndef process_claim(request: ClaimRequest):\n    claim = request.dict()\n    result = process_claim_with_agent(claim)\n    return {"status": "success", "result": result}\n'
(PROJECT_DIR / "app.py").write_text(app_py, encoding="utf-8")
print("Created app.py")

Created app.py


# Step 9: Run and Test Locally

Run:

```bash
uvicorn app:app --reload --port 8000
```

Open:

```text
http://127.0.0.1:8000/docs
```

Test endpoints:

- `GET /health`
- `GET /tools`
- `POST /policy/ask`
- `POST /claim/process`

## Sample Claim for Swagger

```json
{
  "claim_id": "CLM9001",
  "policy_number": "POL1001",
  "customer_name": "Rajesh Sharma",
  "claim_type": "Health",
  "claim_amount": 85000,
  "claim_description": "Hospitalization due to dengue fever.",
  "submitted_documents": [
    "Hospital Bill",
    "Discharge Summary",
    "Doctor Prescription",
    "Claim Form"
  ]
}
```

Expected route:

```text
Approved for Processing
```

# Step 10: Create Dockerfile

Docker packages:

- Python runtime
- FastAPI app
- LangGraph workflow
- RAG service
- MCP tools
- Required libraries
- Startup command

In [ ]:
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
(PROJECT_DIR / "Dockerfile").write_text(dockerfile, encoding="utf-8")
print("Created Dockerfile")

Build and test Docker image:

```bash
docker build -t insurance-claims-capstone .
docker run -p 8000:8000 --env-file .env insurance-claims-capstone
```

Open:

```text
http://127.0.0.1:8000/docs
```

# Step 11: Azure Deployment Commands

## Login

```bash
az login
az account show
```

## Create Resource Group

```bash
az group create \
  --name rg-insurance-claims-capstone \
  --location centralindia
```

## Create Azure Container Registry

```bash
az acr create \
  --resource-group rg-insurance-claims-capstone \
  --name insuranceclaimscapstoneacr \
  --sku Basic
```

## Login to ACR

```bash
az acr login --name insuranceclaimscapstoneacr
```

# Step 12: Push Docker Image to Azure

```bash
docker build -t insurance-claims-capstone .

docker tag insurance-claims-capstone \
insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v1

docker push insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v1

az acr repository list \
  --name insuranceclaimscapstoneacr \
  --output table
```

# Step 13: Create Azure App Service

```bash
az appservice plan create \
  --name asp-insurance-claims-capstone \
  --resource-group rg-insurance-claims-capstone \
  --is-linux \
  --sku B1
```

Create web app:

```bash
az webapp create \
  --resource-group rg-insurance-claims-capstone \
  --plan asp-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --deployment-container-image-name insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v1
```

Project link:

```text
https://insurance-claims-capstone-demo.azurewebsites.net
```

# Step 14: Configure Environment Variables on Azure

Set container port:

```bash
az webapp config appsettings set \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --settings WEBSITES_PORT=8000
```

Set API keys and LangSmith:

```bash
az webapp config appsettings set \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --settings \
  OPENROUTER_BASE_URL="https://openrouter.ai/api/v1" \
  OPENROUTER_MODEL="openai/gpt-4.1-mini" \
  OPENROUTER_API_KEY="your_api_key_here" \
  LANGSMITH_TRACING="true" \
  LANGSMITH_API_KEY="your_langsmith_api_key_here" \
  LANGSMITH_PROJECT="insurance-claims-capstone"
```

# Step 15: Restart and Access Azure Project Link

```bash
az webapp restart \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo
```

Open Swagger:

```text
https://insurance-claims-capstone-demo.azurewebsites.net/docs
```

Health check:

```bash
curl https://insurance-claims-capstone-demo.azurewebsites.net/health
```

# Step 16: Azure Key Vault for Production Secrets

Create Key Vault:

```bash
az keyvault create \
  --name kv-insurance-capstone \
  --resource-group rg-insurance-claims-capstone \
  --location centralindia
```

Store API key:

```bash
az keyvault secret set \
  --vault-name kv-insurance-capstone \
  --name OPENROUTER-API-KEY \
  --value "your_api_key_here"
```

Enable web app identity:

```bash
az webapp identity assign \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo
```

Get principal ID:

```bash
az webapp identity show \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --query principalId \
  --output tsv
```

Give Key Vault permission:

```bash
az keyvault set-policy \
  --name kv-insurance-capstone \
  --object-id YOUR_PRINCIPAL_ID \
  --secret-permissions get list
```

# Step 17: Test Deployed API

```bash
curl -X POST \
  https://insurance-claims-capstone-demo.azurewebsites.net/claim/process \
  -H "Content-Type: application/json" \
  -d '{
    "claim_id": "CLM9001",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Health",
    "claim_amount": 85000,
    "claim_description": "Hospitalization due to dengue fever.",
    "submitted_documents": [
      "Hospital Bill",
      "Discharge Summary",
      "Doctor Prescription",
      "Claim Form"
    ]
  }'
```

Expected response includes:

```text
route: Approved for Processing
risk_level: Low
uipath_result: Claim_Auto_Processing_Robot
```

# Step 18: Connect UiPath to Azure API

In UiPath Studio:

1. Add **HTTP Request** activity.
2. Method: `POST`.
3. Endpoint:

```text
https://insurance-claims-capstone-demo.azurewebsites.net/claim/process
```

4. Header:

```text
Content-Type: application/json
```

5. Body: claim JSON.
6. Deserialize JSON response.
7. Use route to continue automation.

Routes:

```text
Approved for Processing
Rework Required
Human Review
Rejected / Policy Exception
```

# Step 19: Logs and Troubleshooting

Enable logs:

```bash
az webapp log config \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --docker-container-logging filesystem
```

Stream logs:

```bash
az webapp log tail \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo
```

Common issues:

| Issue | Possible Cause | Fix |
|---|---|---|
| App does not start | Wrong port | Set `WEBSITES_PORT=8000` |
| LLM fails | Missing API key | Check App Settings |
| Import error | Missing package | Update requirements.txt |
| Image not found | Wrong ACR image/tag | Check repository and tag |
| Swagger not opening | App still starting | Wait and check logs |

# Step 20: Version Update and Rollback

Create version 2:

```bash
docker build -t insurance-claims-capstone .
docker tag insurance-claims-capstone insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v2
docker push insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v2
```

Update Azure app:

```bash
az webapp config container set \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --container-image-name insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v2
```

Rollback:

```bash
az webapp config container set \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo \
  --container-image-name insuranceclaimscapstoneacr.azurecr.io/insurance-claims-capstone:v1
```

# Step 21: Cost Control

Stop app:

```bash
az webapp stop \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo
```

Start app:

```bash
az webapp start \
  --resource-group rg-insurance-claims-capstone \
  --name insurance-claims-capstone-demo
```

Delete practice resources:

```bash
az group delete \
  --name rg-insurance-claims-capstone \
  --yes \
  --no-wait
```

# Mini Assignments

## Assignment 1
Add property policy `POL4004` with coverage INR 7,00,000.

## Assignment 2
Add a property claim with documents: Damage Photos, Repair Estimate, Ownership Proof, Claim Form.

## Assignment 3
Modify fraud logic: if description contains `duplicate`, set risk as High.

## Assignment 4
Create endpoint `POST /claim/audit` returning only audit report.

## Assignment 5
Deploy updated code as Docker image `v2`.

## Assignment 6
Rollback to Docker image `v1`.

## Assignment 7
Connect UiPath HTTP Request to deployed `/claim/process` endpoint.

# Final Demo Script

1. Show capstone objective.
2. Explain architecture.
3. Show folder structure.
4. Show `.env.example`.
5. Run FastAPI locally.
6. Test `/tools`.
7. Test `/policy/ask`.
8. Test `/claim/process`.
9. Show LangGraph output.
10. Show routing decision.
11. Show mock UiPath trigger.
12. Show audit report.
13. Build Docker image.
14. Push Docker image to Azure.
15. Open deployed Azure URL.
16. Test cloud `/claim/process`.
17. Explain Key Vault and production security.

# Final Checklist

| Task | Done |
|---|---|
| Project folder created | ☐ |
| VS Code environment configured | ☐ |
| `.env` configured | ☐ |
| LangSmith key configured | ☐ |
| requirements.txt created | ☐ |
| MCP tools created | ☐ |
| RAG service created | ☐ |
| LangGraph agent created | ☐ |
| FastAPI endpoints created | ☐ |
| Local Swagger tested | ☐ |
| Docker image built | ☐ |
| Docker container tested locally | ☐ |
| Azure login completed | ☐ |
| Resource group created | ☐ |
| Azure Container Registry created | ☐ |
| Docker image pushed to Azure | ☐ |
| App Service Plan created | ☐ |
| Azure Web App created | ☐ |
| Environment variables configured | ☐ |
| Azure URL tested | ☐ |
| UiPath integration tested | ☐ |
| Logs checked | ☐ |
| Final demo completed | ☐ |

# Summary

You created a capstone project that moves from local AI automation development to cloud deployment:

```text
Insurance Claim Data
   ↓
FastAPI API
   ↓
RAG Policy Assistant
   ↓
LangGraph Multi-Agent Workflow
   ↓
MCP-Style Tools
   ↓
Routing Decision
   ↓
Mock UiPath Trigger
   ↓
Audit Report
   ↓
Azure Deployment
```